<a href="https://colab.research.google.com/github/leejuheon06/Practice_ML_1/blob/main/11_%EC%82%AC%EC%A0%84_%ED%95%99%EC%8A%B5_%EB%AA%A8%EB%8D%B8_%ED%99%9C%EC%9A%A9%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get install -y fonts-nanum* | tail -n 1
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

In [ ]:
# 필요 라이브러리 설치

!pip install torchviz | tail -n 1
!pip install torchinfo | tail -n 1

세션 다시 시작 후 진행

In [ ]:
# 라이브러리 임포트

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 폰트 관련 용도
import matplotlib.font_manager as fm

# 나눔 고딕 폰트의 경로 명시
path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_name = fm.FontProperties(fname=path, size=10).get_name()

In [ ]:
# 파이토치 관련 라이브러리

import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets

In [ ]:
# warning 표시 끄기
import warnings
warnings.simplefilter('ignore')

# 기본 폰트 설정
plt.rcParams['font.family'] = font_name

# 기본 폰트 사이즈 변경
plt.rcParams['font.size'] = 14

# 기본 그래프 사이즈 변경
plt.rcParams['figure.figsize'] = (6,6)

# 기본 그리드 표시
# 필요에 따라 설정할 때는, plt.grid()
plt.rcParams['axes.grid'] = True

# 마이너스 기호 정상 출력
plt.rcParams['axes.unicode_minus'] = False

# 넘파이 부동소수점 자릿수 표시
np.set_printoptions(suppress=True, precision=4)

In [ ]:
# GPU 디바이스 할당

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

공통 함수 불러오기

In [ ]:
# 공통 함수 다운로드
!git clone https://github.com/wikibook/pythonlibs.git

# 공통 함수 불러오기
from pythonlibs.torch_lib1 import *

# 공통 함수 확인
print(README)

11.4 적응형 평균 풀링 함수

In [ ]:
# nn.AdaptiveAvgPool2d 정의
p = nn.AdaptiveAvgPool2d((1,1))
print(p)

# 선형 함수 정의
l1 = nn.Linear(32,10)
print(l1)

In [ ]:
# 사전 학습 모델 시뮬레이션
inputs = torch.randn(100, 32, 16, 16)
m1 = p(inputs)
# print(m1.shape[0])    # 100
m2 = m1.view(m1.shape[0], -1)
m3 = l1(m2)

# shape 확인
print(m1.shape)
print(m2.shape)
print(m3.shape)

11.5 데이터 준비

In [ ]:
# 분류 클래스명 정의

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 분류 클래스 수는 10
n_output = len(classes)
n_output

In [ ]:
# Transforms 정의

# 학습용 데이터 : 정규화에 반전과 RandomErasing 추가
transform_train = transforms.Compose([
    transforms.Resize(112),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.5, scale=(0.02,0.33), ratio=(0.3,3.3), value=0, inplace=False)
])

# 검증용 데이터 : 정규화만 실시
transform = transforms.Compose([
    transforms.Resize(112),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
# 데이터셋
data_root = '/content'

train_set = datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

In [ ]:
# 배치 사이즈 지정
batch_size = 50

# 데이터로더

# 훈련용 데이터 로더
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# 검증용 데이터 로더
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

11.6 ResNet-18 불러오기

In [ ]:
from torchvision import models
net = models.resnet18(pretrained=True)
# pretraind = True로 학습을 마친 파라미터를 동시에 불러오기

In [ ]:
# 모델 개요 표시 1
print(net)

In [ ]:
# 모델 개요 표시 2
net = net.to(device)
summary(net,(100,3,112,112))

In [ ]:
print(net.fc)
print(net.fc.in_features)

11.7 최종 레이어 함수 교체하기

In [ ]:
# 손실 계산 그래프 시각화

criterion = nn.CrossEntropyLoss()
loss = eval_loss(test_loader, device, net, criterion)
g = make_dot(loss, params=dict(net.named_parameters()))
display(g)

11.8 학습과 결과 평가

In [ ]:
# 난수 고정
torch_seed()

# 사전 학습 모델 불러오기
# pretraind = True로 학습을 마친 파라미터도 함께 불러오기
net = models.resnet18(pretrained = True)

# 최종 레이어 함수 입력 차원수 확인
fc_in_features = net.fc.in_features

# 최종 레이어 함수 교체
net.fc = nn.Linear(fc_in_features, n_output)

# GPU 사용
net = net.to(device)

# 학습률
lr = 0.001

# 손실 함수 정의
criterion = nn.CrossEntropyLoss()

# 최적화 함수 정의
optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)

# history 파일 초기화
history = np.zeros((0, 5))

In [ ]:
# 학습
num_epochs = 5
history = fit(net, optimizer, criterion, num_epochs, train_loader, test_loader, device, history)

In [ ]:
# 학습 결과 평가
evaluate_history(history)
# 초기상태 : 손실 : 0.00503  정확도 : 0.91460
# 최종상태 : 손실 : 0.00349 정확도 : 0.93960

In [ ]:
# 이미지와 정답, 예측 결과를 함께 표시
show_images_labels(test_loader, classes, net, device)

11.9 VGG-19-BN 활용하기

In [ ]:
# 사전 학습 모델 불러오기
from torchvision import models
net = models.vgg19_bn(pretrained=True)

In [ ]:
# 모델 개요 표시 1
print(net)

In [ ]:
# 최종 레이어 함수 확인
print(net.classifier[6])

In [ ]:
torch_seed() # 재현성을 위해 난수 다시 고정

# 최종 레이어의 입력 차원 가져오기
in_features = net.classifier[6].in_features
# 새로운 레이어로 교체
net.classifier[6] = nn.Linear(in_features, n_output)
print(net.classifier[6])

net.features = net.features[:-1]
print(net.features)
net.avgpool = nn.Identity()
print(net.avgpool)

In [ ]:
# 모델 개요 표시 2
net = net.to(device)
summary(net,(100,3,112,112))

In [ ]:
# 손실 계산 그래프 시각화

criterion = nn.CrossEntropyLoss()
loss = eval_loss(test_loader, device, net, criterion)
g = make_dot(loss, params=dict(net.named_parameters()))
display(g)

In [ ]:
# 난수 고정
torch_seed()

# 사전 학습 모델 불러오기
net = models.vgg19_bn(pretrained = True)

# 최종 레이어 함수 교체
in_features = net.classifier[6].in_features
net.classifier[6] = nn.Linear(in_features, n_output)

# features 마지막의 MaxPool2d 제거
# >> 더 이상 사전학습된 모델 자체를 훈련하는게 아니니깐
# >> 특징맵의 마지막 레이어의 정보(해상도)를 더 크게 유지하고 싶어서
net.features = net.features[:-1]

# AdaptiveAvgPool2d 제거
# nn.Identity() 단위행렬 예) [[1,0],[0, 1]]
# 아무런 연산 하지 말고 입력받은 거 그래도 통과시켜
net.avgpool = nn.Identity()

# 모델을 GPU로 전송
net = net.to(device)

# 학습률
lr = 0.001

# 손실 함수 정의
criterion = nn.CrossEntropyLoss()

# 최적화 함수 정의
optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)

# history 초기화
history = np.zeros((0, 5))

In [ ]:
# 학습
num_epochs = 5
history = fit(net, optimizer, criterion, num_epochs, train_loader, test_loader, device, history)

In [ ]:
# 결과 요약
evaluate_history(history)
# 초기상태 : 손실 : 0.00373  정확도 : 0.93310
# 최종상태 : 손실 : 0.00263 정확도 : 0.95760

In [ ]:
# 이미지와 정답, 예측 결과를 함께 표시
show_images_labels(test_loader, classes, net, device)

In [ ]:
# eos